**EXPLORATORY DATA ANALYSIS**

In [16]:
import pandas as pd 
import numpy as np

**LAPS**

In [17]:
df_laps = pd.read_parquet(r"C:\Users\Amitava\Downloads\Formula1\All_Data\Bronze_data\laps\all_seasons_laps.parquet")

In [18]:
df_laps.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,lap_number,position,lap_time,lap_time_ms
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,1,1,1:34.233,94233.0
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,1,2,1:35.474,95474.0
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,1,3,1:36.194,96194.0
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,kevin_magnussen,1,4,1:37.107,97107.0
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,max_verstappen,1,5,1:37.684,97684.0


In [19]:
df_laps.shape

(166349, 10)

In [20]:
df_laps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166349 entries, 0 to 166348
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   season        166349 non-null  int64  
 1   round_number  166349 non-null  int64  
 2   race_name     166349 non-null  object 
 3   circuit_ref   166349 non-null  object 
 4   race_date     166349 non-null  object 
 5   driver_ref    166349 non-null  object 
 6   lap_number    166349 non-null  Int16  
 7   position      166343 non-null  Int16  
 8   lap_time      166349 non-null  object 
 9   lap_time_ms   166349 non-null  float64
dtypes: Int16(2), float64(1), int64(2), object(5)
memory usage: 11.1+ MB


In [21]:
df_laps["round_number"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24])

In [22]:
df_laps["race_date"]= pd.to_datetime(df_laps["race_date"])

As there is no **primary key** present therefore we create **candifate key laps_ref** which will act as this table's **primary key** and checking whether combo of driver_ref, season, round_no, lap_number has any duplicates or not

In [23]:
pk_test = df_laps.duplicated(
    subset=["driver_ref", "season", "round_number", "lap_number"]
).sum()
print(f"Duplicates with driver_ref + season + round_number, lap_number: {pk_test}")

Duplicates with driver_ref + season + round_number, lap_number: 0


In [24]:
df_laps["laps_ref"]=df_laps["driver_ref"]+"_"+df_laps["season"].astype(str)+ "_" +df_laps["round_number"].astype(str)+ "_" +df_laps["lap_number"].astype(str)

In [25]:
df_laps[df_laps['position'].isnull()]

,season,round_number,race_name,circuit_ref,race_date,driver_ref,lap_number,position,lap_time,lap_time_ms,laps_ref
13937,2018,14,Italian Grand Prix,monza,2018-09-02,brendon_hartley,6,<NA>,1:27.009,87009.0,brendon_hartley_2018_14_6
14157,2018,14,Italian Grand Prix,monza,2018-09-02,alonso,18,<NA>,1:25.229,85229.0,alonso_2018_14_18
14639,2018,14,Italian Grand Prix,monza,2018-09-02,ricciardo,46,<NA>,1:25.692,85692.0,ricciardo_2018_14_46
17913,2018,18,United States Grand Prix,americas,2018-10-21,alonso,5,<NA>,1:40.933,100933.0,alonso_2018_18_5
18036,2018,18,United States Grand Prix,americas,2018-10-21,grosjean,12,<NA>,1:41.982,101982.0,grosjean_2018_18_12
18648,2018,18,United States Grand Prix,americas,2018-10-21,ricciardo,49,<NA>,1:40.433,100433.0,ricciardo_2018_18_49


In [26]:
df_laps = df_laps.dropna()

We dropped the null columns because the players displayed were already retired from the race but the API still records for following reasons
* **The timing system keeps logging their last known position**
* **The car may have been pushed back to the garage and the transponder kept pinging**
* **Some laps are recorded as the car slowly came back to pit lane**

In [27]:
df_laps = df_laps.sort_values(
    ["season", "round_number", "driver_ref", "lap_number"]
).reset_index(drop=True)

# we have the helper funcs

def ms_to_time(ms):
    """Convert milliseconds to MM:SS.mmm string"""
    if pd.isna(ms):
        return None
    ms = int(ms)
    minutes = ms // 60000
    seconds = (ms % 60000) / 1000
    return f"{minutes}:{seconds:06.3f}"

def ms_to_gap(ms):
    """Convert gap in ms to +SS.mmm or +M:SS.mmm string"""
    if pd.isna(ms):
        return None
    if ms == 0:
        return "+0.000s"
    ms = int(ms)
    minutes = ms // 60000
    seconds = (ms % 60000) / 1000
    if minutes > 0:
        return f"+{minutes}:{seconds:06.3f}"
    else:
        return f"+{seconds:.3f}s"

# cuml, cuml_ms : cummulative sum taken by rider after every lap
df_laps["cuml_ms"] = df_laps.groupby(
    ["season", "round_number", "driver_ref"]
)["lap_time_ms"].cumsum()
df_laps["cuml"] = df_laps["cuml_ms"].apply(ms_to_time)

# Get leader's cumulative time per lap per race
leader_cuml = (
    df_laps[df_laps["position"] == 1][
        ["season", "round_number", "lap_number", "cuml_ms"]
    ]
    .rename(columns={"cuml_ms": "leader_cuml_ms"})
)

# Merge leader time back
df_laps = df_laps.merge(
    leader_cuml,
    on=["season", "round_number", "lap_number"],
    how="left"
)

# gap_to_lead : time gap between leader and current driver 
df_laps["gap_to_lead_ms"] = (
    df_laps["cuml_ms"] - df_laps["leader_cuml_ms"]
).round(1)
df_laps.loc[df_laps["position"] == 1, "gap_to_lead_ms"] = 0.0
df_laps["gap_to_lead"] = df_laps["gap_to_lead_ms"].apply(ms_to_gap)

df_laps = df_laps.drop(columns=["leader_cuml_ms"])

**Added 4 new columns to see cuummulative time and time gap between leader and players**

In [28]:
df_laps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166343 entries, 0 to 166342
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   season          166343 non-null  int64         
 1   round_number    166343 non-null  int64         
 2   race_name       166343 non-null  object        
 3   circuit_ref     166343 non-null  object        
 4   race_date       166343 non-null  datetime64[ns]
 5   driver_ref      166343 non-null  object        
 6   lap_number      166343 non-null  Int16         
 7   position        166343 non-null  Int16         
 8   lap_time        166343 non-null  object        
 9   lap_time_ms     166343 non-null  float64       
 10  laps_ref        166343 non-null  object        
 11  cuml_ms         166343 non-null  float64       
 12  cuml            166343 non-null  object        
 13  gap_to_lead_ms  166343 non-null  float64       
 14  gap_to_lead     166343 non-null  obj

In [29]:
df_laps.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,lap_number,position,lap_time,lap_time_ms,laps_ref,cuml_ms,cuml,gap_to_lead_ms,gap_to_lead
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,alonso,1,10,1:41.528,101528.0,alonso_2018_1_1,101528.0,1:41.528,7295.0,+7.295s
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,alonso,2,10,1:31.565,91565.0,alonso_2018_1_2,193093.0,3:13.093,8595.0,+8.595s
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,alonso,3,10,1:31.304,91304.0,alonso_2018_1_3,284397.0,4:44.397,10090.0,+10.090s
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,alonso,4,10,1:30.551,90551.0,alonso_2018_1_4,374948.0,6:14.948,11168.0,+11.168s
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,alonso,5,10,1:31.910,91910.0,alonso_2018_1_5,466858.0,7:46.858,13917.0,+13.917s


In [30]:
df_laps.to_parquet(r"C:\Users\Amitava\Downloads\Formula1\All_Data\Silver_data\cleaned_laps")